# RAPTOR Chunking Arms — Experiment Runner (Colab)

Compares 5 leaf-chunking arms inside an otherwise **frozen** RAPTOR pipeline, isolating the
chunker as the only variable. Same gpt-oss model (via OpenRouter) is used for summarization
**and** QA across all arms, so results are directly comparable.

| Arm | Chunker |
|---|---|
| `token` | original RAPTOR sentence/token splitter (**baseline RAPTOR**) |
| `structure` | always-LLM structure chunking (char-offset map + per-section repair) |
| `ahc` | **Adaptive Hybrid Chunking** (structure score → route → repair) — *our architecture* |
| `semantic` | SBERT embedding-boundary chunking |
| `flat` | token chunks + FAISS, no tree (contextualizes tree value) |

Datasets & metrics (faithful to arXiv:2401.18059v1): **QASPER** → Answer-F1; **QuALITY** →
accuracy (+ HARD); **NarrativeQA** → ROUGE-L / BLEU-1/4 / METEOR. Retrieval = collapsed tree,
2000-token budget.

> **Tip:** the LLM runs over the API, so a GPU runtime is optional (only SBERT embeddings use it).
> Run the small **smoke test** first to validate the whole pipeline. Every LLM call is cached on
> disk, so reruns are cheap and the run is resumable.

## 1. Clone the repo and install the full stack

In [ ]:
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

## 2. Credentials + METEOR data
Get a free key at https://openrouter.ai/keys .

In [ ]:
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

import nltk
for pkg in ('wordnet', 'omw-1.4', 'punkt'):
    nltk.download(pkg, quiet=True)
print('ready')

## 2b. Persist cache + results to Google Drive (recommended)

Free-tier Colab sessions recycle, and OpenRouter free-tier rate limits mean a full run spans many sittings. Mounting Drive keeps `.llm_cache/` and `results/` across disconnects, so every rerun **resumes from cache** instead of re-spending LLM calls. Authorize Drive access when prompted. (Off Colab this falls back to a local `./raptor_runs`.)

In [ ]:
# Persist the LLM cache + results to Google Drive so a disconnect never loses
# the (expensive) cached calls. Re-running then resumes from cache.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/raptor_runs'
except Exception as e:
    print('Not on Colab / Drive unavailable -> using local ./raptor_runs', e)
    RUN_DIR = 'raptor_runs'

import os
CACHE_DIR = os.path.join(RUN_DIR, '.llm_cache')
RESULTS_DIR = os.path.join(RUN_DIR, 'results')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('RUN_DIR   =', RUN_DIR)
print('CACHE_DIR =', CACHE_DIR)

## 3. (Optional) Smoke-test the harness offline
All logic is unit-tested without network/model loads; run it to confirm the clone is intact.

In [ ]:
!python -m pytest tests/ -q

## 4. Configure — start with the fast smoke test

This first config is deliberately tiny: **`token` (baseline RAPTOR) vs `ahc` (our method)** on **QuALITY only**, **2 docs**, 1 seed. QuALITY is short multiple-choice, so this exercises the whole API path end-to-end (build tree → summarize → retrieve → answer → score) in minutes — *before* you commit hours to the full grid. Scale up in section 8 once this is green.

Caching is keyed by `(model, prompt)`, so nothing here is wasted: the calls this run makes are reused by the bigger runs.

In [ ]:
from experiments.config import ExperimentConfig

# FAST SMOKE TEST — token (baseline RAPTOR) vs ahc (our method) on QuALITY, 2 docs.
cfg = ExperimentConfig(
    model='openai/gpt-oss-120b:free',   # -> 'openai/gpt-oss-120b' (paid) if rate-limited
    arms=['token', 'ahc'],
    datasets=['quality'],
    subset_sizes={'quality': 2},
    seeds=[0],
    retrieval_max_tokens=2000,          # paper's collapsed-tree main setting
    leaf_max_tokens=100,
    tau=0.5,                            # AHC routing threshold
    cache_dir=CACHE_DIR,                # on Drive (section 2b)
    results_dir=RESULTS_DIR,
)
cfg.to_dict()

## 4b. Verify dataset alignment with the paper (run once)

Confirms QuALITY is scored the paper's way — **accuracy over the dev split, plus the QuALITY-HARD subset** — and that gold labels + the HARD flag actually populated from the HF mirror. A silent field-name mismatch would otherwise make accuracy read **0**; the `assert` below fails loudly instead. (QuALITY-HARD = questions a majority of annotators got wrong under time pressure — the dataset's `difficult` flag.)

In [ ]:
from experiments.datasets import get_loader

ql = get_loader('quality').load(limit=5)          # dev/validation split
qs = [q for d in ql for q in d.questions]
with_gold = sum(q.gold_index is not None for q in qs)
hard = sum(q.is_hard for q in qs)
print(f'QuALITY dev: {len(ql)} docs, {len(qs)} questions')
print(f'  with gold label : {with_gold}/{len(qs)}   (must be > 0)')
print(f'  HARD (difficult): {hard}/{len(qs)}')
print(f'  options/question: {sorted({len(q.options or []) for q in qs})}   -> chance acc = 0.25')
assert with_gold > 0, 'No gold labels parsed -> accuracy would be a silent 0. Check the mirror schema.'
print('alignment OK: accuracy + QuALITY-HARD will compute correctly.')

## 5. Run
Builds one tree per (arm, dataset, doc) — SBERT embeddings + gpt-oss summaries — then answers
every question with the same gpt-oss reader and scores it. Progress prints per dataset/arm.
Interrupting is safe: cached calls make a rerun resume.

In [ ]:
from experiments import runner, report

results = runner.run(cfg, seed=cfg.seeds[0])
print('records:', len(results['records']))

## 6. Results: per-arm tables + AHC routing rate

In [ ]:
agg = report.aggregate(results['records'])
routing = report.routing_rate(results['records'])
print(report.to_markdown(agg, routing))

### Reading your numbers against the RAPTOR paper

Your reader is `gpt-oss-120b`, **not** the paper's GPT-4 / GPT-3 / UnifiedQA, so *absolute* numbers will not match a leaderboard — this is a **controlled arm-vs-arm** study (like the paper's own Tables 2 & 4), where only the chunker changes. Use the paper as a sanity band, not a target. The question that matters: does `ahc` move the metric vs `token` on the **same split, same reader**?

**QuALITY — accuracy (chance = 25%).** This harness reports the **dev** split (reproducible; the test set needs a leaderboard submission).

| Setting (source) | Acc |
|---|---:|
| BM25 + UnifiedQA — dev (Table 4) | 49.9 |
| DPR + UnifiedQA — dev (Table 4) | 53.9 |
| RAPTOR + UnifiedQA-3B — dev (Table 4) | 56.6 |
| RAPTOR + GPT-3 — dev (Table 4) | 62.4 |
| RAPTOR + GPT-4 — **test** set (Table 7) | 82.6 |
| RAPTOR + GPT-4 — **test**, QuALITY-HARD (Table 7) | 76.2 |

> The 82.6 / 76.2 headline is GPT-4 on the **hidden test set** — do **not** compare your dev number to it directly. The comparable rows are the dev ones (56.6 / 62.4).

**QASPER** — Answer token-F1 (RAPTOR): UnifiedQA 36.6 · GPT-3 53.1 · GPT-4 55.7 (Tables 3 & 5).
**NarrativeQA** — RAPTOR + UnifiedQA-3B (Table 6): ROUGE-L 30.8 · BLEU-1 23.5 · BLEU-4 6.4 · METEOR 19.1.

## 7. Resuming after a disconnect

Because the cache + results already live on Drive (section 2b), recovery is automatic: reconnect, re-run sections **1, 2, 2b, 4, 5, 6**, and every already-completed LLM call is served from `CACHE_DIR` — only the missing calls hit the network. Stop and resume as many times as the free tier forces you to.

In [ ]:
# Nothing to copy if you mounted Drive in section 2b — cache + results persist there.
# Sanity-check what has accumulated so far:
import os
print('cached LLM calls:', len(os.listdir(CACHE_DIR)) if os.path.isdir(CACHE_DIR) else 0)
print('results files:   ', os.listdir(RESULTS_DIR) if os.path.isdir(RESULTS_DIR) else [])

## 8. Scale up to the full study

Re-run sections **4 → 5 → 6** with a bigger config. Keep `cache_dir=CACHE_DIR` / `results_dir=RESULTS_DIR` so results accumulate on Drive. On the free tier, run **one arm or one dataset at a time** — the cache makes this seamless.

**Pilot (all 5 arms, 3 docs/dataset) — validates every arm + metric:**
```python
cfg = ExperimentConfig(
    model='openai/gpt-oss-120b:free',
    arms=['token', 'structure', 'ahc', 'semantic', 'flat'],
    datasets=['qasper', 'quality', 'narrativeqa'],
    subset_sizes={'qasper': 3, 'quality': 3, 'narrativeqa': 3},
    seeds=[0],
    cache_dir=CACHE_DIR, results_dir=RESULTS_DIR,
)
results = runner.run(cfg, seed=0)
print(report.to_markdown(report.aggregate(results['records']),
                         report.routing_rate(results['records'])))
```

**Full study (paper subsets, 3 seeds → mean±std + bootstrap CIs):**
```python
cfg = ExperimentConfig(
    model='openai/gpt-oss-120b',   # paid; the free tier will rate-limit a run this size
    subset_sizes={'qasper': 50, 'quality': 50, 'narrativeqa': 25},
    seeds=[0, 1, 2],
    cache_dir=CACHE_DIR, results_dir=RESULTS_DIR,
)
for s in cfg.seeds:
    runner.run(cfg, seed=s)
```

> **Reality check:** the full 5-arm × 3-dataset × 3-seed grid is thousands of LLM calls. The `:free` model *will* rate-limit it into many sittings (the cache makes that survivable). For a run you want finished in one or two sessions, swap to the paid `openai/gpt-oss-120b` — one line, same results.